In [18]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score


from catboost import CatBoostRegressor

In [3]:
train_sample_path = "../train_sample.csv"
test_sample_path = "../test_sample.csv"

In [4]:
train_sample = pd.read_csv(train_sample_path)
train_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor,travel_time
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,NaN,9,1,NaN,high,NaN,1,0.878909,26.907612
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,NaN,7,1,medium,high,NaN,1,1.081668,27.489129


In [5]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,8,1,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,9,1,low,medium,fog,2,1.121015


In [6]:
start = train_sample["start_point"].str.split().str[0]
end = train_sample["end_point"].str.split().str[0]

train_sample["start_end_point"] = start + " " + end
test_sample["start_end_point"] = test_sample["start_point"].str.split().str[0] + " " + test_sample["end_point"].str.split().str[0]

In [7]:
train_sample['start_end_point'].value_counts()

start_end_point
Central West     4153
West South       4048
Central South    4016
South East       4013
North West       4010
Central East     3987
Central North    3964
North East       3948
North South      3935
West East        3926
Name: count, dtype: int64

Based on EDA on missing_values_handling.ipynb.

if start_point == 'Central' then vehicle_density = 'high'


else vehicle_density = 'medium'

In [8]:
train_sample["vehicle_density"] = (
    train_sample["vehicle_density"]
    .fillna(
        train_sample["start_end_point"]
        .str.split()
        .str[0]
        .eq("Central")
        .map({True: "high", False: "medium"})
    )
)

In [9]:
test_sample["vehicle_density"] = (
    test_sample["vehicle_density"]
    .fillna(
        test_sample["start_end_point"]
        .str.split()
        .str[0]
        .eq("Central")
        .map({True: "high", False: "medium"})
    )
)

In [10]:
# Numeric → mean
numeric_cols = train_sample.select_dtypes(include='number').columns

train_sample[numeric_cols] = train_sample[numeric_cols].fillna(
    train_sample[numeric_cols].mean()
)

# Categorical → mode
categorical_cols = train_sample.select_dtypes(
    include=['str']
).columns

for col in categorical_cols:
    train_sample[col] = train_sample[col].fillna(
        train_sample[col].mode()[0]
    )

train_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
travel_time                      0
start_end_point                  0
dtype: int64

In [11]:
# Numeric → mean
numeric_cols = test_sample.select_dtypes(include='number').columns

test_sample[numeric_cols] = test_sample[numeric_cols].fillna(
    test_sample[numeric_cols].mean()
)

# Categorical → mode
categorical_cols = test_sample.select_dtypes(
    include=['str']
).columns

for col in categorical_cols:
    test_sample[col] = test_sample[col].fillna(
        test_sample[col].mode()[0]
    )

test_sample.isnull().sum()
test_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
start_end_point                  0
dtype: int64

In [12]:
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              40000 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                40000 non-null  str    
 8   population_density             40000 non-null  str    
 9   weather                        40000 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
 1

In [13]:
test_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    3000 non-null   str    
 1   end_point                      3000 non-null   str    
 2   time_of_day                    3000 non-null   str    
 3   day_of_week                    3000 non-null   str    
 4   traffic_condition              3000 non-null   float64
 5   event_count                    3000 non-null   int64  
 6   is_holiday                     3000 non-null   int64  
 7   vehicle_density                3000 non-null   str    
 8   population_density             3000 non-null   str    
 9   weather                        3000 non-null   str    
 10  public_transport_availability  3000 non-null   int64  
 11  historical_delay_factor        3000 non-null   float64
 12  start_end_point                3000 non-null   object 
dtyp

In [14]:
X_test = test_sample

In [15]:
y_train = train_sample['travel_time']
X_train = train_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"]]
X_test = test_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"]]
# etc.
# your code here

In [16]:
cat_cols = [
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [17]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0785634	total: 188ms	remaining: 56.1s
100:	learn: 4.7968389	total: 4.82s	remaining: 9.5s
200:	learn: 4.7264195	total: 9.74s	remaining: 4.8s
299:	learn: 4.7016782	total: 14.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0942582	total: 54.5ms	remaining: 16.3s
100:	learn: 4.7751020	total: 4.44s	remaining: 8.74s
200:	learn: 4.7138742	total: 8.83s	remaining: 4.35s
299:	learn: 4.6930824	total: 13.1s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 12.9575673	total: 36.3ms	remaining: 10.9s
100:	learn: 4.5948477	total: 4.19s	remaining: 8.25s
200:	learn: 4.5201115	total: 8.36s	remaining: 4.12s
299:	learn: 4.4895734	total: 12.9s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0897340	total: 48.2ms	remaining: 14.4s
100:	learn: 4.7833751	total: 4.46s	remaining: 8.8s
200:	learn: 4.7052689	total: 9.15s	remaining: 4.5s
299:	learn: 4.6793404	total: 14.3s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0651410	total: 35.

In [21]:
model.fit(X_train, y_train)

Learning rate set to 0.195167
0:	learn: 12.9564298	total: 55.9ms	remaining: 16.7s
100:	learn: 4.7233779	total: 4.65s	remaining: 9.16s
200:	learn: 4.6579847	total: 9.68s	remaining: 4.77s
299:	learn: 4.6235468	total: 14.6s	remaining: 0us


CatBoostRegressor(cat_features=('start_point', 'end_point', 'time_of_day', 'day_of_week', 'vehicle_density', 'population_density', 'weather', 'is_holiday', 'start_end_point', 'public_transport_availability'), depth=3, iterations=300, loss_function='RMSE', random_seed=38, verbose=100)

In [22]:
y_train_hat = model.predict(X_train)
mse_lr = mean_squared_error(y_train_hat, y_train)
r2_lr = r2_score(y_train_hat, y_train)
mse_lr, r2_lr

(21.023408811579163, 0.8945998543848099)

In [23]:
y_hat_test = model.predict(X_test)
pd.DataFrame(y_hat_test).to_csv('submission.csv', index=False)